In [ ]:
!pip -q install kagglehub

import os, re, glob, random, math, time, subprocess, tarfile, zipfile, warnings
import numpy as np, pandas as pd
import requests
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda"

DATASET = "busi"
INCLUDE_NORMAL = True
SEED = 42
S = 256
BATCH = 16
EPOCHS = 100 if DATASET == "busi" else 50
LR, WD = 1e-3, 1e-2
SPLIT = (0.7, 0.1, 0.2)
LAMBDA_CLS, LAMBDA_CAM, LAMBDA_ACE = 0.2, 0.3, 0.5
M_PRIORS = 5
PRIOR_RES = 64
ACE_REGION = 4
N_BINS = 10

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print(device, DATASET)

In [ ]:
def to_square(a):
    h, w = a.shape; m = max(h, w); out = np.zeros((m, m), a.dtype)
    y0, x0 = (m - h) // 2, (m - w) // 2; out[y0:y0 + h, x0:x0 + w] = a; return out

def prep(im, m):
    img = np.asarray(Image.fromarray(to_square(im)).resize((S, S), Image.BILINEAR))
    msk = np.asarray(Image.fromarray(to_square(m.astype(np.uint8))).resize((S, S), Image.NEAREST))
    return img, msk

def load_busi():
    import kagglehub
    root = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
    imgs, msks, strat = [], [], []
    for dp, dn, fs in sorted(os.walk(root)):
        name = os.path.basename(dp).lower()
        if name not in ("benign", "malignant", "normal"): continue
        if name == "normal" and not INCLUDE_NORMAL: continue
        for f in sorted(fs):
            if not f.lower().endswith(".png") or "_mask" in f.lower(): continue
            im = np.asarray(Image.open(os.path.join(dp, f)).convert("L"))
            m = np.zeros(im.shape, bool)
            for mp in glob.glob(os.path.join(glob.escape(dp), glob.escape(os.path.splitext(f)[0]) + "_mask*.png")):
                mm = np.asarray(Image.open(mp).convert("L")) > 127
                if mm.shape == m.shape: m |= mm
            a, b = prep(im, m); imgs.append(a); msks.append(b); strat.append(name)
    msks = np.stack(msks)
    return np.stack(imgs), msks, msks.reshape(len(msks), -1).any(1).astype(int), np.array(strat)

def load_pu():
    ART_ID, FILE_ID = 32672274, 65547621
    DATA = "/content/pu2756"; os.makedirs(DATA, exist_ok=True)
    def extract(path, dst):
        n = path.lower()
        if n.endswith(".zip"): zipfile.ZipFile(path).extractall(dst)
        elif n.endswith((".tar", ".tar.gz", ".tgz", ".tar.bz2")): tarfile.open(path).extractall(dst)
        elif n.endswith(".7z"):
            subprocess.run(["pip", "-q", "install", "py7zr"], check=True); import py7zr; py7zr.SevenZipFile(path).extractall(dst)
        elif n.endswith(".rar"):
            subprocess.run("apt-get -qq install -y unrar", shell=True); subprocess.run(["unrar", "x", "-o+", path, DATA + "/"])
        else: return False
        return True
    if not glob.glob(DATA + "/**/*.*", recursive=True):
        files = requests.get(f"https://api.figshare.com/v2/articles/{ART_ID}", timeout=60).json()["files"]
        for f in [f for f in files if f["id"] == FILE_ID] or files:
            dst = os.path.join(DATA, f["name"])
            with requests.get(f["download_url"], stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(dst, "wb") as fh:
                    for chunk in r.iter_content(1 << 20): fh.write(chunk)
            if extract(dst, DATA): os.remove(dst)
    all_files = sorted(p for p in glob.glob(DATA + "/**/*", recursive=True) if os.path.isfile(p))
    pics = [p for p in all_files if p.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"))]
    tok = re.compile(r"(mask|label|gt|annot|ground)", re.I)
    is_mask = lambda p: bool(tok.search(os.path.relpath(os.path.dirname(p), DATA))) or bool(tok.search(os.path.splitext(os.path.basename(p))[0]))
    norm = lambda p: re.sub(r"([_\-\s]?(mask|label|seg|gt|annotation)s?)+$", "", os.path.splitext(os.path.basename(p))[0].lower())
    mb = {}
    for p in pics:
        if is_mask(p): mb.setdefault(norm(p), p)
    pairs = [(p, mb[norm(p)]) for p in pics if not is_mask(p) and norm(p) in mb]
    assert pairs, "no image/mask pairs found"
    print(len(pairs), "pairs | example:", os.path.relpath(pairs[0][0], DATA), "<->", os.path.relpath(pairs[0][1], DATA))
    paths = [os.path.relpath(ip, DATA).lower() for ip, _ in pairs]
    lab = [1 if "malignant" in r else 0 if "benign" in r else None for r in paths]
    if any(v is None for v in lab):
        lab = None
        stems = {norm(ip) for ip, _ in pairs}
        for tp in [p for p in all_files if p.lower().endswith((".csv", ".xlsx", ".xls"))]:
            try: df = pd.read_csv(tp) if tp.lower().endswith(".csv") else pd.read_excel(tp)
            except Exception: continue
            for nc in df.columns:
                keys = df[nc].astype(str).map(norm)
                if keys.isin(stems).mean() < 0.5: continue
                for lc in [c for c in df.columns if c != nc and re.search(r"malig|benign|patholog|diagnos|label|class|type|nature", str(c), re.I)]:
                    col = df[lc]
                    if col.nunique() != 2: continue
                    pos = (col == col.max()) if pd.api.types.is_numeric_dtype(col) else col.astype(str).str.contains("malig", case=False)
                    d = dict(zip(keys, pos.astype(int)))
                    if all(norm(ip) in d for ip, _ in pairs):
                        print("labels from", os.path.basename(tp), nc, lc, col.value_counts().to_dict())
                        lab = [d[norm(ip)] for ip, _ in pairs]; break
                if lab is not None: break
            if lab is not None: break
    imgs, msks = [], []
    for ip, mp in tqdm(pairs):
        im = np.asarray(Image.open(ip).convert("L")); m = np.asarray(Image.open(mp).convert("L")); m = (m > 127) if m.max() > 1 else (m > 0)
        if m.shape != im.shape: m = np.asarray(Image.fromarray(m.astype(np.uint8) * 255).resize(im.shape[::-1], Image.NEAREST)) > 127
        a, b = prep(im, m); imgs.append(a); msks.append(b)
    msks = np.stack(msks)
    if lab is None:
        print("no benign/malignant label found, using mask non-empty as image-level target")
        lab = msks.reshape(len(msks), -1).any(1).astype(int)
    return np.stack(imgs), msks, np.array(lab), np.array(lab).astype(str)

imgs_np, msks_np, cls_np, strat_np = load_busi() if DATASET == "busi" else load_pu()
IMGS, MSKS, CLS = torch.from_numpy(imgs_np), torch.from_numpy(msks_np), torch.from_numpy(cls_np).float()
print(IMGS.shape, "positives:", int(CLS.sum()), "| classes:", pd.Series(strat_np).value_counts().to_dict(), "| mean lesion fraction: %.3f" % MSKS.float().mean())

rng = np.random.RandomState(SEED); split_idx = {"train": [], "val": [], "test": []}
for sv in np.unique(strat_np):
    ids = np.where(strat_np == sv)[0]; rng.shuffle(ids); n = len(ids)
    a, b = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
    split_idx["train"] += ids[:a].tolist(); split_idx["val"] += ids[a:b].tolist(); split_idx["test"] += ids[b:].tolist()
arrs = {k: (IMGS[v], MSKS[v], CLS[v]) for k, v in split_idx.items()}
for k, a in arrs.items(): print(k, len(a[0]), "positives:", int(a[2].sum()))

def random_affine(x, y):
    ang, sc = math.radians(random.uniform(-10, 10)), random.uniform(0.9, 1.1)
    tx, ty = random.uniform(-0.08, 0.08), random.uniform(-0.08, 0.08)
    c_, s_ = math.cos(ang) / sc, math.sin(ang) / sc
    grid = F.affine_grid(torch.tensor([[[c_, -s_, tx], [s_, c_, ty]]], dtype=torch.float32), (1, 1, S, S), align_corners=False)
    return (F.grid_sample(x[None], grid, mode="bilinear", align_corners=False)[0],
            F.grid_sample(y[None], grid, mode="nearest", align_corners=False)[0])

class SegDS(Dataset):
    def __init__(self, a, train=False): self.imgs, self.masks, self.cls = a; self.train = train
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float().div(255.)[None]; y = self.masks[i].float()[None]
        if self.train:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5: x, y = random_affine(x, y)
            x = x * (1 + 0.2 * (random.random() - 0.5)) + 0.1 * (random.random() - 0.5)
        x = (x - x.mean()) / (x.std() + 1e-6)
        return x, y, self.cls[i], i

train_ds, val_ds, test_ds = SegDS(arrs["train"], True), SegDS(arrs["val"]), SegDS(arrs["test"])
mk = lambda ds, sh: DataLoader(ds, BATCH, shuffle=sh, num_workers=2, drop_last=sh, pin_memory=True)
train_loader, val_loader, test_loader = mk(train_ds, True), mk(val_ds, False), mk(test_ds, False)

In [ ]:
class GRN(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, C, 1, 1)); self.beta = nn.Parameter(torch.zeros(1, C, 1, 1))
    def forward(self, x):
        gx = torch.linalg.vector_norm(x.float(), dim=(2, 3), keepdim=True)
        nx = (gx / (gx.mean(1, keepdim=True) + 1e-6)).to(x.dtype)
        return self.gamma * (x * nx) + self.beta + x

class MedNeXtBlock(nn.Module):
    def __init__(self, cin, cout, exp, k=3, mode="same"):
        super().__init__()
        self.mode = mode; mid = cin * exp
        if mode == "down":
            self.dw = nn.Conv2d(cin, cin, k, 2, k // 2, groups=cin); self.res = nn.Conv2d(cin, cout, 1, 2)
        elif mode == "up":
            self.dw = nn.ConvTranspose2d(cin, cin, k, 2, k // 2, output_padding=1, groups=cin); self.res = nn.ConvTranspose2d(cin, cout, 1, 2, output_padding=1)
        else:
            self.dw = nn.Conv2d(cin, cin, k, 1, k // 2, groups=cin); self.res = None
        self.norm = nn.GroupNorm(cin, cin)
        self.expand = nn.Conv2d(cin, mid, 1); self.act = nn.GELU(); self.grn = GRN(mid); self.compress = nn.Conv2d(mid, cout, 1)
    def forward(self, x):
        y = self.compress(self.grn(self.act(self.expand(self.norm(self.dw(x))))))
        return y + (x if self.res is None else self.res(x))

class MedNeXt2D(nn.Module):
    def __init__(self, use_cls=False, use_cam=False, C=32, k=3, n=2, exp=(2, 3, 4, 4, 4, 4, 4, 3, 2)):
        super().__init__()
        assert use_cls or not use_cam
        self.use_cls, self.use_cam = use_cls, use_cam
        ch = [C, C * 2, C * 4, C * 8, C * 16]
        self.stem = nn.Conv2d(1, C, 1)
        self.enc = nn.ModuleList([nn.Sequential(*[MedNeXtBlock(ch[i], ch[i], exp[i], k) for _ in range(n)]) for i in range(4)])
        self.down = nn.ModuleList([MedNeXtBlock(ch[i], ch[i + 1], exp[i], k, "down") for i in range(4)])
        self.bott = nn.Sequential(*[MedNeXtBlock(ch[4], ch[4], exp[4], k) for _ in range(n)])
        order = (3, 2, 1, 0)
        self.up = nn.ModuleList([MedNeXtBlock(ch[i + 1], ch[i], exp[5 + 3 - i], k, "up") for i in order])
        self.dec = nn.ModuleList([nn.Sequential(*[MedNeXtBlock(ch[i], ch[i], exp[5 + 3 - i], k) for _ in range(n)]) for i in order])
        if use_cls: self.fc = nn.Linear(ch[4], 1)
        if use_cam: self.refine = nn.Sequential(nn.Conv2d(C + 1, C, 3, 1, 1), nn.GELU())
        self.head = nn.Conv2d(C, 1, 1)
    def forward(self, x):
        size = x.shape[-2:]; x = self.stem(x); skips = []
        for e, d in zip(self.enc, self.down):
            x = e(x); skips.append(x); x = d(x)
        bott = self.bott(x); d = bott
        for j, i in enumerate((3, 2, 1, 0)):
            d = self.dec[j](self.up[j](d) + skips[i])
        out = {}
        if self.use_cls:
            logit = self.fc(bott.mean((2, 3)))
            cam = F.relu(torch.einsum("bchw,c->bhw", bott, self.fc.weight[0])).unsqueeze(1)
            cam = cam / (cam.amax((2, 3), keepdim=True) + 1e-6)
            cam = F.interpolate(cam, size=size, mode="bilinear", align_corners=False)
            out["cls"], out["cam"] = logit, cam
            if self.use_cam: d = self.refine(torch.cat([d, cam * torch.sigmoid(logit).view(-1, 1, 1, 1)], 1))
        out["logits"] = self.head(d)
        return out

_m = MedNeXt2D(True, True).to(device)
with torch.no_grad(): _o = _m(torch.randn(2, 1, S, S, device=device))
print({k: tuple(v.shape) for k, v in _o.items()}, "params: %.1fM" % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o

In [ ]:
def dice_loss(logits, y, smooth=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return (1 - (2 * inter + smooth) / (den + smooth)).mean()

def seg_loss(logits, y):
    return 0.5 * dice_loss(logits, y) + 0.5 * F.binary_cross_entropy_with_logits(logits, y)

def cam_loss(cam, y, c):
    pos = c > 0.5
    if pos.sum() == 0: return cam.sum() * 0
    return F.binary_cross_entropy(cam[pos].clamp(1e-4, 1 - 1e-4), y[pos])

class PriorBank:
    def __init__(self, masks, res=PRIOR_RES):
        self.res = res
        self.bank = F.adaptive_avg_pool2d(masks[:, None].float(), res).flatten(1).to(device)
        self.norm = F.normalize(self.bank, dim=1)
    @torch.no_grad()
    def retrieve(self, prob, M, exclude=None):
        q = F.adaptive_avg_pool2d(prob, self.res).flatten(1)
        sim = F.normalize(q, dim=1) @ self.norm.T
        if exclude is not None: sim[torch.arange(len(q), device=sim.device), exclude.to(sim.device)] = -1
        w, idx = sim.topk(M, dim=1)
        return w.clamp(0, 1), self.bank[idx]

def uncertainty(prob, bank, M=M_PRIORS, exclude=None):
    prob = prob.float()
    w, priors = bank.retrieve(prob.detach(), M, exclude)
    q = F.adaptive_avg_pool2d(prob, bank.res).flatten(1)
    U = (w[:, :, None] * (priors - q[:, None]).abs()).sum(1) / M
    return U.view(-1, 1, bank.res, bank.res)

def ace_loss(U, prob, y, res=PRIOR_RES, region=ACE_REGION):
    e = (F.adaptive_avg_pool2d(prob, res) - F.adaptive_avg_pool2d(y, res)).abs().detach()
    return (F.avg_pool2d(U, region) - F.avg_pool2d(e, region)).abs().mean()

bank = PriorBank(arrs["train"][1])
print(tuple(bank.bank.shape))

In [ ]:
def autocast(): return torch.autocast("cuda", enabled=AMP)

@torch.no_grad()
def val_dice(model, loader):
    model.eval(); ds = []
    for x, y, c, i in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        pb = (torch.sigmoid(out["logits"].float()) > 0.5).float()
        inter = (pb * y).sum((1, 2, 3))
        ds.append((2 * inter + 1) / (pb.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1))
    return torch.cat(ds).mean().item()

@torch.no_grad()
def evaluate(model, loader, bank, n_vis=0):
    model.eval()
    per = {k: [] for k in ("dice", "iou", "prec", "rec", "haslesion")}
    TP = FP = FN = 0.0; nb = N_BINS
    cnt = torch.zeros(2, nb, device=device, dtype=torch.float64); sconf = torch.zeros_like(cnt); sacc = torch.zeros_like(cnt)
    g = torch.zeros(6, dtype=torch.float64, device=device)
    uerr = uok = nerr = nok = 0.0
    pcorr, img_u, img_e, vis, cp, ct = [], [], [], [], [], []
    for x, y, c, idx in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        prob = torch.sigmoid(out["logits"].float()); pb = (prob > 0.5).float()
        inter = (pb * y).sum((1, 2, 3)); ps = pb.sum((1, 2, 3)); ys = y.sum((1, 2, 3))
        dice = (2 * inter + 1) / (ps + ys + 1)
        per["dice"].append(dice); per["iou"].append((inter + 1) / (ps + ys - inter + 1))
        per["prec"].append((inter + 1) / (ps + 1)); per["rec"].append((inter + 1) / (ys + 1)); per["haslesion"].append(ys > 0)
        TP += inter.sum().item(); FP += (ps - inter).sum().item(); FN += (ys - inter).sum().item()
        if "cls" in out: cp += torch.sigmoid(out["cls"].float()).squeeze(1).tolist(); ct += c.tolist()
        for k, (conf, ind) in enumerate([(prob, y), (1 - prob, 1 - y)]):
            b = (conf * nb).long().clamp(0, nb - 1).flatten()
            cnt[k] += torch.bincount(b, minlength=nb).double()
            sconf[k] += torch.bincount(b, weights=conf.flatten().double(), minlength=nb)
            sacc[k] += torch.bincount(b, weights=ind.flatten().double(), minlength=nb)
        U = uncertainty(prob, bank)
        err = F.adaptive_avg_pool2d((pb != y).float(), bank.res)
        u, e = U.flatten(1).double(), err.flatten(1).double()
        uc, ec = u - u.mean(1, keepdim=True), e - e.mean(1, keepdim=True)
        den = uc.norm(dim=1) * ec.norm(dim=1); ok = den > 1e-8
        pcorr += ((uc * ec).sum(1)[ok] / den[ok]).tolist()
        has = e > 0
        uerr += u[has].sum().item(); nerr += has.sum().item(); uok += u[~has].sum().item(); nok += (~has).sum().item()
        g += torch.stack([torch.tensor(float(u.numel()), device=device, dtype=torch.float64), u.sum(), e.sum(), (u * u).sum(), (e * e).sum(), (u * e).sum()])
        img_u += u.mean(1).tolist(); img_e += (1 - dice).tolist()
        if len(vis) < n_vis:
            cam = out.get("cam", None)
            for j in range(len(x)):
                if len(vis) >= n_vis: break
                vis.append(dict(x=x[j, 0].cpu(), y=y[j, 0].cpu(), p=prob[j, 0].cpu(),
                                cam=None if cam is None else cam[j, 0].float().cpu(),
                                U=F.interpolate(U[j:j + 1], size=x.shape[-2:], mode="bilinear")[0, 0].cpu()))
    n, su, se, suu, see, sue = g.tolist()
    cov = sue / n - (su / n) * (se / n)
    gcorr = cov / (math.sqrt(max(suu / n - (su / n) ** 2, 1e-12)) * math.sqrt(max(see / n - (se / n) ** 2, 1e-12)))
    valid = cnt > 0
    o, ex = sacc / cnt.clamp(min=1), sconf / cnt.clamp(min=1)
    ace = (((o - ex).abs() * valid).sum() / valid.sum()).item()
    m = {k: torch.cat(v) for k, v in per.items()}
    hl = m["haslesion"]
    res = {"Dice": m["dice"].mean().item(), "Dice_lesion_only": m["dice"][hl].mean().item() if hl.any() else float("nan"),
           "IoU": m["iou"].mean().item(), "Precision": m["prec"].mean().item(), "Recall": m["rec"].mean().item(),
           "F1": 2 * TP / (2 * TP + FP + FN + 1e-9), "ACE": ace,
           "U_mean": su / n, "U_on_error": uerr / max(nerr, 1), "U_on_correct": uok / max(nok, 1),
           "corr_U_err_pixel": gcorr, "corr_U_err_per_image": float(np.mean(pcorr)) if pcorr else float("nan"),
           "corr_meanU_vs_(1-Dice)": float(np.corrcoef(img_u, img_e)[0, 1])}
    if cp:
        res["cls_acc"] = float(np.mean((np.array(cp) > 0.5) == (np.array(ct) > 0.5)))
        res["cls_auc"] = float(roc_auc_score(ct, cp)) if len(set(ct)) > 1 else float("nan")
    return res, vis

In [ ]:
CKPT_DIR = "/content/ckpt_mednext"; os.makedirs(CKPT_DIR, exist_ok=True)

VARIANTS = {
    "MedNeXt":             (False, False, False),
    "MedNeXt+CH":          (True,  False, False),
    "MedNeXt+CH+CAM":      (True,  True,  False),
    "MedNeXt+CH+CAM+ACE":  (True,  True,  True),
}

def train_variant(name, use_ch, use_cam, use_ace):
    set_seed(SEED)
    model = MedNeXt2D(use_ch, use_cam).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, path, hist = -1, f"{CKPT_DIR}/{name}.pt", []
    for ep in range(EPOCHS):
        for gp in opt.param_groups: gp["lr"] = LR * (1 - ep / EPOCHS) ** 0.9
        model.train(); agg = {"tot": 0, "seg": 0, "cls": 0, "cam": 0, "ace": 0}; nit = 0
        for x, y, c, idx in tqdm(train_loader, desc=f"{name} ep{ep + 1}/{EPOCHS}", leave=False):
            x, y, c = x.to(device), y.to(device), c.to(device)
            with autocast(): out = model(x)
            logits = out["logits"].float(); prob = torch.sigmoid(logits)
            Ls = seg_loss(logits, y); L = Ls; Lc = Lm = La = torch.zeros((), device=device)
            if use_ch:
                Lc = F.binary_cross_entropy_with_logits(out["cls"].float().squeeze(1), c); L = L + LAMBDA_CLS * Lc
            if use_cam:
                Lm = cam_loss(out["cam"].float(), y, c); L = L + LAMBDA_CAM * Lm
            if use_ace:
                U = uncertainty(prob, bank, M_PRIORS, exclude=idx)
                La = ace_loss(U, prob, y); L = L + LAMBDA_ACE * La
            opt.zero_grad(set_to_none=True)
            scaler.scale(L).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            scaler.step(opt); scaler.update()
            for k, v in zip(agg, (L, Ls, Lc, Lm, La)): agg[k] += float(v)
            nit += 1
        vd = val_dice(model, val_loader)
        hist.append({"epoch": ep + 1, "val_dice": vd, **{k: v / nit for k, v in agg.items()}})
        print(f"[{name}] ep{ep + 1:03d} loss {agg['tot'] / nit:.4f} (seg {agg['seg'] / nit:.3f} cls {agg['cls'] / nit:.3f} cam {agg['cam'] / nit:.3f} ace {agg['ace'] / nit:.3f}) | val Dice {vd:.4f}")
        if vd > best: best = vd; torch.save(model.state_dict(), path)
    model.load_state_dict(torch.load(path))
    return model, pd.DataFrame(hist)

models, histories, results, vis_store = {}, {}, {}, {}
for name, flags in VARIANTS.items():
    t0 = time.time()
    models[name], histories[name] = train_variant(name, *flags)
    results[name], vis_store[name] = evaluate(models[name], test_loader, bank, n_vis=8)
    print(f"== {name}: {(time.time() - t0) / 60:.1f} min | test Dice {results[name]['Dice']:.4f}\n")

In [ ]:
res_df = pd.DataFrame(results).T
seg_cols = ["Dice", "Dice_lesion_only", "IoU", "Precision", "Recall", "F1"]
unc_cols = ["ACE", "U_mean", "U_on_error", "U_on_correct", "corr_U_err_pixel", "corr_U_err_per_image", "corr_meanU_vs_(1-Dice)"]
display(res_df[seg_cols].round(4)); display(res_df[unc_cols].round(4))
if "cls_auc" in res_df: display(res_df[["cls_acc", "cls_auc"]].dropna().round(4))
res_df.to_csv(f"/content/{DATASET}_mednext_ablation_results.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for n_, h in histories.items():
    ax[0].plot(h.epoch, h.seg, label=n_); ax[1].plot(h.epoch, h.val_dice, label=n_)
ax[0].set_title("train L_seg"); ax[1].set_title("val Dice"); ax[1].legend(fontsize=7)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
def show_qualitative(name, n=6):
    v = vis_store[name][:n]
    cols = ["Image", "Ground truth", "Prediction", "CAM", "Uncertainty U", "Error |pred-GT|"]
    fig, axs = plt.subplots(len(v), 6, figsize=(14, 2.3 * len(v))); axs = np.atleast_2d(axs)
    for r, s in enumerate(v):
        axs[r, 0].imshow(s["x"], cmap="gray")
        axs[r, 1].imshow(s["x"], cmap="gray"); axs[r, 1].imshow(s["y"], cmap="Greens", alpha=0.45)
        axs[r, 2].imshow(s["x"], cmap="gray"); axs[r, 2].imshow((s["p"] > 0.5).float(), cmap="Reds", alpha=0.45)
        axs[r, 3].imshow(s["x"], cmap="gray")
        if s["cam"] is not None: axs[r, 3].imshow(s["cam"], cmap="jet", alpha=0.5, vmin=0, vmax=1)
        im = axs[r, 4].imshow(s["U"], cmap="magma"); plt.colorbar(im, ax=axs[r, 4], fraction=0.046)
        axs[r, 5].imshow((s["p"] > 0.5).float().sub(s["y"]).abs(), cmap="gray", vmin=0, vmax=1)
        for a in axs[r]: a.axis("off")
    for j, t in enumerate(cols): axs[0, j].set_title(t, fontsize=9)
    fig.suptitle(name); plt.tight_layout(); plt.show()

show_qualitative("MedNeXt+CH+CAM+ACE")